In [1]:
import os
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
from tqdm import tqdm
from multiprocessing import Pool
import logging
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

In [2]:
# Initialize MediaPipe FaceLandmarker
def initialize_landmarker():
    base_options = mp.tasks.BaseOptions(model_asset_path='/home/mojtaba.kananisarche/Zh/UTA/Process/face_landmarker_v2_with_blendshapes.task')
    options = mp.tasks.vision.FaceLandmarkerOptions(
        base_options=base_options,
        output_face_blendshapes=True,
        output_facial_transformation_matrixes=False,
        num_faces=1
    )
    return mp.tasks.vision.FaceLandmarker.create_from_options(options)

In [3]:
def draw_landmarks_on_frame(frame, detection_result, indices=list(range(478))):
    """
    Draws face landmarks on a single video frame.

    Parameters:
    - frame: numpy.ndarray
        The video frame as a NumPy array.
    - detection_result: FaceLandmarkerResult
        The detection result containing face landmarks.

    Returns:
    - annotated_frame: numpy.ndarray
        The frame with face landmarks drawn.
    """
    annotated_frame = frame.copy()
    for face_landmarks in detection_result.face_landmarks:
        for i in range(len(face_landmarks)):
            if i not in indices:
                continue
            landmark = face_landmarks[i]
            x = int(landmark.x * annotated_frame.shape[1])
            y = int(landmark.y * annotated_frame.shape[0])
            cv2.circle(annotated_frame, (x, y), radius=2, color=(0, 255, 0), thickness=-1)
    return annotated_frame

In [4]:
def process_video(video_path, save=False, start_second=0, end_second=None, use_tqdm=True, indices=list(range(478))):
    """
    Processes a video, detects landmarks in each frame within the specified range, 
    and optionally saves the annotated video.

    Parameters:
    - video_path: str
        Path to the input video.
    - save: bool or str
        Whether to save the annotated video. If a string, it specifies the output path.
    - start_second: int
        Start time in seconds for processing the video.
    - end_second: int or None
        End time in seconds for processing the video. If None, processes until the end of the video.
    - use_tqdm: bool
        Whether to use a progress bar.

    Returns:
    - blendshape_results: list
        A list of detected blendshape scores for each frame.
    """
    detector = initialize_landmarker()
    
    logging.info(f"Starting processing for video: {video_path}")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        logging.warning(f"Failed to open video: {video_path}")
        return []

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    start_frame = int(start_second * fps)
    end_frame = int(end_second * fps) if end_second is not None else total_frames

    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    blendshape_results = []
    frames = []
    detection_results = []
    

    with tqdm(total=end_frame - start_frame, desc=f"Processing {os.path.basename(video_path)}", unit="frame", disable=not use_tqdm) as progress_bar:
        while cap.isOpened() and cap.get(cv2.CAP_PROP_POS_FRAMES) < end_frame:
            ret, frame = cap.read()
            
            if cap.get(cv2.CAP_PROP_POS_FRAMES) % 1000 == 0:
                logging.info(f"frame number {cap.get(cv2.CAP_PROP_POS_FRAMES)} in video: {video_path}")
                
            if not ret:
                logging.warning(f"Failed to read frame at position {cap.get(cv2.CAP_PROP_POS_FRAMES)} in video: {video_path}")
                break
            
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
            detection_result = detector.detect(mp_image)
            detection_results.append(detection_result)
     
            frame_blendshapes = [
                {blendshape.category_name: blendshape.score for blendshape in face_blendshapes}
                for face_blendshapes in detection_result.face_blendshapes
            ]
            blendshape_results.append(frame_blendshapes)
            frames.append(frame)

            progress_bar.update(1)

    cap.release()
    logging.info(f"Completed processing for video: {video_path}")

    if save:
        filename = os.path.basename(os.path.splitext(video_path)[0]) + "_annotated" + os.path.splitext(video_path)[1]
        folder = save if isinstance(save, str) else os.path.dirname(video_path)
        output_path = os.path.join(folder, filename)
        logging.info(f"Saving annotated video to: {output_path}")

        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
        
        with tqdm(total=end_frame - start_frame, desc=f"Saving {os.path.basename(video_path)}", unit="frame", disable=not use_tqdm) as progress_bar:    
            for frame, detection_result in zip(frames, detection_results):
                annotated_frame = draw_landmarks_on_frame(frame, detection_result, indices)
                out.write(annotated_frame)

                progress_bar.update(1)

        out.release()
        logging.info(f"Annotated video saved at: {output_path}")

    return blendshape_results, detection_results


In [5]:
# Wrapper function for processing a single video
def process_video_wrapper(args):
    return process_single_video(*args)

def process_single_video(video_file, input_folder, output_folder, save=False, start_second=0, end_second=None, use_tqdm=False, indices=list(range(478))):
    input_path = os.path.join(input_folder, video_file)
    output_csv = os.path.join(output_folder, os.path.splitext(video_file)[0] + "_blendshape.csv")

    if os.path.exists(output_csv):
        logging.info(f"Skipping already processed video: {input_path}")
        return None

    try:
        blendshape_results, detection_results = process_video(input_path, save, start_second, end_second, use_tqdm, indices)
        keys = next((frame[0].keys() for frame in blendshape_results if frame), [])

        pd.DataFrame([
            {key: round(value, 2) if isinstance(value, float) else value for key, value in frame[0].items()} if frame else {key: 0 for key in keys}
            for frame in blendshape_results
        ]).to_csv(output_csv, index=False)

        logging.info(f"Saved blendshape results to: {output_csv}")
        
        return [blendshape_results, detection_results]
    
    except Exception as e:
        logging.error(f"Error processing {video_file}: {e}")
        return None
    

In [9]:
def process_videos_and_save_parallel(input_folder, output_folder, save=False, start_second=0, end_second=None, use_tqdm=False, indices=list(range(478))):
    """
    Process all videos in a folder, create a DataFrame from blendshape_results, and save as CSV.
    """
    os.makedirs(output_folder, exist_ok=True)
    video_files = [f for f in os.listdir(input_folder) if f.lower().endswith(('.mp4', '.mov', '.avi', '.MOV'))]

    args_list = [(video_file, input_folder, output_folder, save, start_second, end_second, use_tqdm, indices) for video_file in video_files]
    logging.info(args_list)
    
    for args in args_list:
        print(args)
        process_video_wrapper(args)
#     with ProcessPoolExecutor(15) as executor:
#         executor.map(process_video_wrapper, args_list)

In [10]:
input_folder = "/work/long_lab/Mojtaba/data/DROZY/DROZY/videos_i8/"
output_folder = "/work/long_lab/Mojtaba/data/DROZY/DROZY/blendshapes/"
process_videos_and_save_parallel(input_folder, output_folder)

('1-1.mp4', '/work/long_lab/Mojtaba/data/DROZY/DROZY/videos_i8/', '/work/long_lab/Mojtaba/data/DROZY/DROZY/blendshapes/', False, 0, None, False, [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192,

KeyboardInterrupt: 

In [ ]:
left_eye_upper_indices = [466, 388, 387, 386, 385, 384, 398]
left_eye_lower_indices = [263, 249, 390, 373, 374, 380, 381, 382, 362]
left_eye_indices = left_eye_upper_indices + left_eye_lower_indices
right_eye_upper_indices = [246, 161, 160, 159, 158, 157, 173]
right_eye_lower_indices = [33, 7, 163, 144, 145, 153, 154, 155, 133]
right_eye_indices = right_eye_upper_indices + right_eye_lower_indices
lips_upper_outer= [61, 185, 40, 39, 37, 0, 267, 269, 270, 409, 291]
lips_lower_outer= [146, 91, 181, 84, 17, 314, 405, 321, 375, 291]
lips_upper_inner= [78, 191, 80, 81, 82, 13, 312, 311, 310, 415, 308]
lips_lower_inner= [78, 95, 88, 178, 87, 14, 317, 402, 318, 324, 308]
middle_line_indices= [10,9,6,4,1,152]
nose_tip_index = 1

In [120]:
result = process_single_video(video_file = "1_01_0_30fps.mov", 
                             input_folder = "/home/mojtaba.kananisarche/Zh/UTA/Video/Raw_30fps/", 
                             output_folder = "/home/mojtaba.kananisarche/Zh/UTA/Temp/", 
                             save = "/home/mojtaba.kananisarche/Zh/UTA/Temp/", 
                             use_tqdm = True)

Saving 1_01_0_30fps.mov: 100%|██████████| 30/30 [00:00<00:00, 57.97frame/s]


In [75]:
blendshape_results, detection_results = result

In [ ]:
"/work/long_lab/Mojtaba/data/DROZY/DROZY/blendshapes/"